# Plan with CLIP-ViT Planner on MiniGrid

Load a trained RSSM world model and use a CLIP-ViT-based planner to select actions via heuristic candidate sampling and discounted cosine-similarity scoring.

## 1. Setup

In [ ]:
!pip install gymnasium minigrid matplotlib pyyaml Pillow tqdm open-clip-torch imageio -q

import sys
!git clone https://github.com/mihalko711/tbank_intro_problem.git
sys.path.insert(0, 'tbank_intro_problem')
%cd tbank_intro_problem

## 2. Imports, config, env, checkpoint

In [ ]:
import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm

from src import (
    RSSMWorldModel, collect_episode, evaluate,
    get_env_properties, make_minigrid_env, seed_everything,
)
from src.planner import CLIPScorer, Planner, HeuristicCandidates, UniformCandidates, CEMCandidates
from src.environment import action_to_env

CHECKPOINT_PATH = "/kaggle/input/tbank-wm-checkpoint/MiniGrid-Empty-6x6-v0_default_final.pth"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

with open('configs/minigrid_default.yml') as f:
    config = yaml.safe_load(f)
rssm_cfg = config['rssm']
seed_everything(config['seed'])

env = make_minigrid_env(config['environment_name'], seed=config['seed'])
env_eval = make_minigrid_env(config['environment_name'], seed=config['seed'] + 999)
obs_shape, action_size = get_env_properties(env)
print(f'Obs shape: {obs_shape}, action size: {action_size}')

rssm = RSSMWorldModel(obs_shape, action_size, rssm_cfg, device)
rssm.load_checkpoint(CHECKPOINT_PATH)
print(f'Checkpoint loaded: {CHECKPOINT_PATH}')

## 3. Create CLIP-ViT Planner

In [ ]:
clip_scorer = CLIPScorer(device, rssm)
planner = Planner(
    rssm, clip_scorer,
    num_candidates=64,
    horizon=rssm_cfg['imagination_horizon'],
    gamma=0.99,
    score_mode='discounted_sum',
)
print('Planner (CLIP-ViT) ready')
print(f'Heuristic candidates: {planner.num_candidates} x {planner.horizon} steps')

## 4. Visualise heuristic candidates

Take a starting observation from a random episode, decode the imagined trajectories for the best, median, and worst candidate.

In [ ]:
from src.utils import symexp

@torch.no_grad()
def visualize_candidates(rssm, planner, env, action_size, n_cols=15, save_path=None):
    rec, lat = rssm.reset_state()
    action = torch.zeros(1, action_size, device=rssm.device)
    obs = env.reset()
    rec = rssm.recurrent_model(rec, lat, action)
    encoded = rssm.encoder(torch.from_numpy(obs).float().unsqueeze(0).to(rssm.device))
    lat, _ = rssm.posterior_net(torch.cat((rec, encoded.view(1, -1)), -1))

    candidates = planner._heuristic_sampler.sample()
    trajectories = rssm.imagine_rollouts(rec, lat, candidates)
    scores = clip_scorer.score_rollouts(trajectories, planner.gamma).reshape(-1)

    order = scores.argsort(descending=True)
    idxs = [order[0].item(), order[len(order)//2].item(), order[-1].item()]
    labels = ['Best', 'Median', 'Worst']

    h = min(n_cols, planner.horizon)
    fig, axes = plt.subplots(3, h, figsize=(h * 2, 6))
    for row, (idx, label) in enumerate(zip(idxs, labels)):
        states = trajectories[idx, :h]
        decoded = symexp(rssm.decoder(states)).clamp(0, 1).cpu().numpy()
        for t in range(h):
            axes[row, t].imshow(np.transpose(decoded[t], (1, 2, 0)))
            axes[row, t].axis('off')
            if t == 0:
                axes[row, t].set_title(f'{label}\nscore={scores[idx]:.2f}', fontsize=9)

    plt.suptitle(f'Decoded prior rollouts (64 candidates, heuristic 80/10/10)', fontsize=11)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show()


os.makedirs('plots', exist_ok=True)
visualize_candidates(rssm, planner, env, action_size, n_cols=15, save_path='plots/plan_candidates.png')

## 5. Define all strategies

Build a dictionary of 15 strategies: CLIP (Heuristic RS, Uniform RS, CEM) × (agg, argmax) × (discounted_sum, max), Reward × (argmax, agg) × (discounted_sum, max), and Random.

In [ ]:
strategies = {}

_rs = rssm.recurrent_size

for sm in ["discounted_sum", "max"]:
    for sel in ["agg", "argmax"]:
        name = f"CLIP Heuristic RS {sel} {sm}"
        strategies[name] = {
            "action_fn": lambda full_state, obs, _sm=sm, _sel=sel, _r=_rs: planner.plan_action_random_shooting(
                full_state[:, :_r], full_state[:, _r:], use_heuristic=True, use_argmax=(_sel=="argmax"), score_mode=_sm
            ),
            "score_fn": lambda traj, gamma, _sm=sm: clip_scorer.score_rollouts(traj, gamma, mode=_sm),
        }

for sm in ["discounted_sum", "max"]:
    for sel in ["agg", "argmax"]:
        name = f"CLIP Uniform RS {sel} {sm}"
        strategies[name] = {
            "action_fn": lambda full_state, obs, _sm=sm, _sel=sel, _r=_rs: planner.plan_action_random_shooting(
                full_state[:, :_r], full_state[:, _r:], use_heuristic=False, use_argmax=(_sel=="argmax"), score_mode=_sm
            ),
            "score_fn": lambda traj, gamma, _sm=sm: clip_scorer.score_rollouts(traj, gamma, mode=_sm),
        }

for sm in ["discounted_sum", "max"]:
    name = f"CLIP CEM {sm}"
    strategies[name] = {
        "action_fn": lambda full_state, obs, _sm=sm, _r=_rs: planner.plan_action_cem(
            full_state[:, :_r], full_state[:, _r:], score_mode=_sm
        ),
        "score_fn": lambda traj, gamma, _sm=sm: clip_scorer.score_rollouts(traj, gamma, mode=_sm),
    }

for sm in ["discounted_sum", "max"]:
    for sel in ["argmax", "agg"]:
        name = f"Reward {sel} {sm}"
        if sel == "argmax":
            strategies[name] = {
                "action_fn": lambda full_state, obs, _sm=sm, _r=_rs: planner.plan_action_reward(
                    full_state[:, :_r], full_state[:, _r:], score_mode=_sm
                ),
                "score_fn": lambda traj, gamma, _sm=sm: planner._score_rollouts_reward(traj, gamma, mode=_sm),
            }
        else:
            strategies[name] = {
                "action_fn": lambda full_state, obs, _sm=sm, _r=_rs: planner.plan_action_reward_aggregated(
                    full_state[:, :_r], full_state[:, _r:], score_mode=_sm
                ),
                "score_fn": lambda traj, gamma, _sm=sm: planner._score_rollouts_reward(traj, gamma, mode=_sm),
            }

strategies["Random"] = {
    "action_fn": lambda full_state, obs: torch.nn.functional.one_hot(
        torch.tensor(np.random.choice(getattr(env_eval, 'valid_actions', lambda: list(range(action_size)))()), device=device).unsqueeze(0),
        action_size
    ).float(),
    "score_fn": None,
}

print(f"Defined {len(strategies)} strategies:")
for k in strategies:
    print(f"  {k}")

# Demo: run one episode with the first non-random strategy
demo_name = next(k for k in strategies if k != "Random")
demo_score, demo_steps = collect_episode(env, rssm, rssm.buffer, action_fn=strategies[demo_name]["action_fn"])
print(f'{demo_name} episode: reward={demo_score:.2f}, steps={demo_steps}')

In [ ]:
# Grab the last `demo_steps` observations directly from the buffer
end = rssm.buffer._index
obs_seq = rssm.buffer.observations[end - demo_steps : end]
rewards_seq = rssm.buffer.rewards[end - demo_steps : end].squeeze(-1)

fig, axes = plt.subplots(2, len(obs_seq), figsize=(len(obs_seq) * 1.5, 3))
if len(obs_seq) == 1:
    axes = axes.reshape(2, 1)
for t in range(len(obs_seq)):
    axes[0, t].imshow(np.transpose(obs_seq[t], (1, 2, 0)))
    axes[0, t].axis('off')
    if t == 0:
        axes[0, t].set_title('Observations', fontsize=8)
    axes[1, t].text(0.5, 0.5, f'{rewards_seq[t]:.1f}', ha='center', va='center', fontsize=8)
    axes[1, t].axis('off')
    if t == 0:
        axes[1, t].set_title('Reward', fontsize=8)
plt.suptitle(f'{demo_name} episode: total reward = {demo_score:.2f}, steps = {demo_steps}')
plt.tight_layout()
plt.show()

## 6. Evaluation: all 15 strategies

Evaluate all 15 strategies and compare episode returns.

In [ ]:
@torch.no_grad()
def evaluate(env, rssm, action_fn, num_episodes=10):
    scores = []
    it = tqdm(range(num_episodes), desc='Eval')
    for _ in it:
        rec, lat = rssm.reset_state()
        action = torch.zeros(1, rssm.action_size, device=rssm.device)
        obs = env.reset()
        score = 0
        done = False
        while not done:
            rec = rssm.recurrent_model(rec, lat, action)
            encoded = rssm.encoder(
                torch.from_numpy(obs).float().unsqueeze(0).to(rssm.device)
            )
            lat, _ = rssm.posterior_net(
                torch.cat((rec, encoded.view(1, -1)), -1)
            )
            full_state = torch.cat((rec, lat), -1)
            action = action_fn(full_state, obs)
            action_numpy = action.cpu().numpy().reshape(-1)
            next_obs, reward, done = env.step(action_to_env(action_numpy))
            obs = next_obs
            score += reward
        scores.append(score)
        it.set_postfix(score=f'{score:.2f}')
    return np.mean(scores), np.std(scores)

num_ep = 10

results = {}
for name, cfg in tqdm(strategies.items(), desc='Evaluating'):
    avg, std = evaluate(env_eval, rssm, cfg["action_fn"], num_episodes=num_ep)
    results[name] = (avg, std)
    print(f'{name:40s}  {avg:.3f} +/- {std:.3f}')

In [ ]:
names = list(results.keys())
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]
colors = [plt.cm.tab10(i % 10) for i in range(len(names))]

fig, ax = plt.subplots(figsize=(18, 5))
bars = ax.bar(range(len(names)), means, yerr=stds, capsize=4, color=colors)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Episode reward')
ax.set_title(f'Evaluation over {num_ep} episodes — all {len(strategies)} strategies')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/plan_15_strategies.png', bbox_inches='tight', dpi=120)
plt.show()

## 7. Distribution of CLIP scores across candidates

For several environment steps, show the histogram of CLIP cosine-similarity scores over all 64 heuristic candidates.

In [ ]:
@torch.no_grad()
def collect_score_distribution(rssm, planner, observation, action, n_steps=5):
    rec, lat = rssm.reset_state()
    enc = rssm.encoder(torch.from_numpy(observation).float().unsqueeze(0).to(rssm.device))
    rec = rssm.recurrent_model(rec, lat, action)
    lat, _ = rssm.posterior_net(torch.cat((rec, enc.view(1, -1)), -1))
    candidates = planner._heuristic_sampler.sample()
    trajectories = rssm.imagine_rollouts(rec, lat, candidates)
    scores = clip_scorer.score_rollouts(trajectories, planner.gamma)
    return scores.cpu().numpy()


obs = env_eval.reset()
all_scores = []
act = torch.zeros(1, action_size, device=device)
for step in range(8):
    scores = collect_score_distribution(rssm, planner, obs, act)
    all_scores.append(scores)
    act = planner_policy(torch.zeros(1, rssm.recurrent_size + rssm.latent_size, device=device), obs)
    obs, reward, done = env_eval.step(
        int(torch.argmax(act.cpu().view(-1)).item())
    )

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    if i < len(all_scores):
        ax.hist(all_scores[i], bins=20, color='purple', alpha=0.7)
        ax.set_title(f'Step {i}')
        ax.set_xlabel('CLIP score')
        ax.axvline(all_scores[i].max(), color='red', ls='--', label=f'best={all_scores[i].max():.2f}')
        ax.legend(fontsize=7)
plt.suptitle('Distribution of CLIP scores across 64 heuristic candidates per step')
plt.tight_layout()
plt.savefig('plots/plan_score_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

## 8. Decision GIFs for all strategies

Generate a decision GIF for every strategy, showing the imagined rollouts and scores.

In [ ]:
import imageio
from src.utils import symexp
from IPython.display import clear_output


@torch.no_grad()
def make_decision_frame(rssm, planner, obs, rec, lat, action_size,
                         score_fn=None, action_names=None, chosen_action=None, step=0):
    if action_names is None:
        action_names = ['Left', 'Right', 'Forward']

    if score_fn is None:
        fig, ax = plt.subplots(1, 1, figsize=(3, 3))
        ax.imshow(np.transpose(obs, (1, 2, 0)))
        ax.set_title(f'Step {step}\nRandom', fontsize=9)
        ax.axis('off')
        plt.tight_layout()
        fig.canvas.draw()
        img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
        plt.close(fig)
        return img

    candidates = planner._heuristic_sampler.sample()
    trajectories = rssm.imagine_rollouts(rec, lat, candidates)
    scores = score_fn(trajectories, planner.gamma).reshape(-1)

    first_actions = candidates[:, 0].argmax(dim=-1)
    agg_scores = []
    best_candidates = []
    for a in range(action_size):
        mask = first_actions == a
        if mask.any():
            agg_scores.append(scores[mask].mean().item())
            best_in_group = scores[mask].argmax()
            best_candidates.append(trajectories[mask][best_in_group, :5])
        else:
            agg_scores.append(-float('inf'))
            best_candidates.append(None)

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

    axes[0].imshow(np.transpose(obs, (1, 2, 0)))
    axes[0].set_title(f'Step {step}\nReal observation', fontsize=9)
    axes[0].axis('off')

    best_a = max(range(action_size), key=lambda i: agg_scores[i])

    for a in range(action_size):
        ax = axes[a + 1]
        if best_candidates[a] is not None:
            decoded = symexp(rssm.decoder(best_candidates[a])).clamp(0, 1).cpu().numpy()
            strip = np.concatenate([np.transpose(d, (1, 2, 0)) for d in decoded], axis=1)
            ax.imshow(strip)
        ax.axis('off')
        chosen = ' ← chosen' if (chosen_action is not None and a == chosen_action) or (chosen_action is None and a == best_a) else ''
        ax.set_title(f"{action_names[a]}: {agg_scores[a]:.2f}{chosen}", fontsize=9)
        ax.set_xlabel('Imagined steps →', fontsize=7)

    plt.tight_layout()
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)
    return img


os.makedirs('visualizations', exist_ok=True)

for idx, (name, cfg) in enumerate(tqdm(strategies.items(), desc='Generating GIFs')):
    frames = []
    rec, lat = rssm.reset_state()
    action = torch.zeros(1, action_size, device=device)
    obs = env_eval.reset()
    done = False
    step = 0
    max_steps = 30
    action_names = ['Left', 'Right', 'Forward']

    while not done and step < max_steps:
        rec = rssm.recurrent_model(rec, lat, action)
        encoded = rssm.encoder(torch.from_numpy(obs).float().unsqueeze(0).to(device))
        lat, _ = rssm.posterior_net(torch.cat((rec, encoded.view(1, -1)), -1))
        full_state = torch.cat((rec, lat), -1)

        img = make_decision_frame(
            rssm, planner, obs, rec, lat, action_size,
            score_fn=cfg["score_fn"], action_names=action_names, step=step,
        )
        frames.append(img)

        action = cfg["action_fn"](full_state, obs)
        act_np = action.cpu().numpy().reshape(-1)
        obs, reward, done = env_eval.step(action_to_env(act_np))
        step += 1

    safe_name = name.lower().replace(' ', '_')
    gif_path = f'visualizations/decision_{safe_name}.gif'
    imageio.mimsave(gif_path, frames, fps=2, loop=0)
    clear_output(wait=True)
    print(f'[{idx + 1}/{len(strategies)}] Saved: {gif_path} ({step} frames)')

print('All GIFs generated.')

## 9. Cleanup


In [ ]:
env.close()
env_eval.close()
print('Done.')